In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 15,
    "legend.fontsize": 10,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 2.1,
    "axes.linewidth": 1.1,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,
})


# Shared baseline parameters
phi0 = 0.8
Dc = 1.0
D_c = 1.0
kdec = 0.1
a = 1.0

phi_m = 0.9
PeR = 1e-3
kappa = 5.0
chi0_mips = 0.5

default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


# Bulk chemical-potential slope
def dmu_h(phi):
    return (
        1 / phi
        - 2
        - 0.6 * phi
        - (4 / np.pi) * PeR * phi_m * (phi - 2 * phi_m) / (phi - phi_m) ** 2
    )

dmu = dmu_h(phi0)


def add_panel_label(ax, label, x=-0.16, y=1.03):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="normal",
        va="top",
        ha="left",
        clip_on=False
    )


# Dispersion relations
def omega_classical_ks_plus(q, Dphi, chi):
    A = Dphi * q**2
    B = Dc * q**2 + kdec
    C = a * chi * phi0 * q**2
    disc = (A - B)**2 + 4 * C
    return -0.5 * (A + B) + 0.5 * np.sqrt(disc)

def omega_autocrine_ks_plus(q, Dphi):
    omega_phi = -Dphi * q**2
    omega_c = -(Dc * q**2 + kdec * phi0)
    return np.maximum(omega_phi, omega_c)

def omega_autocrine_mips_plus(q, M0):
    omega_phi = -M0 * q**2 * (dmu + kappa * q**2)
    omega_c = -(Dc * q**2 + kdec * phi0)
    return np.maximum(omega_phi, omega_c)

def omega_plus_dim(q, M0, k_val):
    M = M0 * q**2 * (dmu + kappa * q**2)
    D = D_c * q**2 + k_val * phi0
    disc = (M - D)**2
    return 0.5 * (-(M + D) + np.sqrt(disc.astype(complex)))

alpha0_base = 2.0
Da0_base = 0.5
PeC_base = 0.5

def omega_pm(q, alpha0, Da0, PeC, a_val=1):
    A = phi0 * q**2 * (dmu + q**2)
    B = PeC * phi0 * q**2
    C = a_val
    D = (q**2 + Da0) / alpha0

    disc = (A - D) ** 2 + 4 * B * C
    root = np.sqrt(disc.astype(complex))

    omega_plus = 0.5 * (-(A + D) + root)
    omega_minus = 0.5 * (-(A + D) - root)

    return omega_plus, omega_minus


fig = plt.figure(figsize=(14, 8.5))

gs = GridSpec(
    nrows=2,
    ncols=3,
    width_ratios=[1.45, 1.0, 1.0],
    height_ratios=[1.0, 1.0],
    wspace=0.40,
    hspace=0.45,
)

axA = fig.add_subplot(gs[:, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[0, 2])
axD = fig.add_subplot(gs[1, 1])
axE = fig.add_subplot(gs[1, 2])


# a: Main mechanism comparison
q_main = np.linspace(1e-4, 0.8, 1200)

Dphi_ks = 0.2
chi_ks = 3.0
Dphi_auto = 0.2
M0_mips = 2.0

w_ks = omega_classical_ks_plus(q_main, Dphi=Dphi_ks, chi=chi_ks)
w_auto = omega_autocrine_ks_plus(q_main, Dphi=Dphi_auto)
w_mips = omega_autocrine_mips_plus(q_main, M0=M0_mips)

axA.plot(q_main, w_ks, color=default_colors[0], label="Classical KS")
axA.plot(q_main, w_auto, color=default_colors[1], label="Autocrine KS")
axA.plot(q_main, w_mips, color=default_colors[2], label="Autocrine MIPS")

axA.axhline(0, ls=":", color="grey", lw=1.2)
axA.set_xlim(0, 0.8)
axA.set_xlabel(r"wavenumber $q$")
axA.set_ylabel(r"growth rate $\mathrm{Re}\ \omega$")
axA.legend(loc="upper left", frameon=False)

# Inset in panel a: wider q range
q_wide = np.linspace(1e-4, 6.0, 2500)

w_ks_wide = omega_classical_ks_plus(q_wide, Dphi=Dphi_ks, chi=chi_ks)
w_auto_wide = omega_autocrine_ks_plus(q_wide, Dphi=Dphi_auto)
w_mips_wide = omega_autocrine_mips_plus(q_wide, M0=M0_mips)

axins = axA.inset_axes([0.10, 0.05, 0.42, 0.34])

axins.plot(q_wide, w_ks_wide, color=default_colors[0], lw=1.5)
axins.plot(q_wide, w_auto_wide, color=default_colors[1], lw=1.5)
axins.plot(q_wide, w_mips_wide, color=default_colors[2], lw=1.5)

axins.axhline(0, ls=":", color="grey", lw=0.9)
axins.set_xlim(0, 6.0)
axins.set_ylim(-5.0, 1.4)
axins.tick_params(axis="both", labelsize=9, width=0.8, length=3)


# b: Autocrine MIPS varying alpha0
q_auto = np.linspace(1e-3, 0.6, 600)

M0_list = [2, 4, 6, 8]
k_fixed = 0.1

for M0, col in zip(M0_list, default_colors[:len(M0_list)]):
    w = omega_plus_dim(q_auto, M0=M0, k_val=k_fixed)
    axB.plot(q_auto, w.real, color=col, lw=1.8, label=rf"$\alpha_0 = {M0}$")
    axB.plot(q_auto, w.imag, "--", color=col, lw=1.0)

axB.axhline(0, ls=":", color="grey", lw=1.0)
axB.set_xlim(0, 0.6)
axB.set_ylim(-0.5, 0.6)
axB.set_xlabel(r"wavenumber $q$")
axB.set_ylabel(r"growth rate $\omega$")
axB.legend(loc="lower left", 
           bbox_to_anchor=(0.03, 0.03), 
           ncol=2, frameon=True, 
           framealpha=0.75, edgecolor="none", columnspacing=0.8, handlelength=1.6, handletextpad=0.5)


# c: Autocrine MIPS varying Da0
M0_fixed = 2.0
k_vals = [0.02, 0.06, 0.1, 0.14]
Da0_list = [0.1, 0.3, 0.5, 0.7]

for k_val, Da0, col in zip(k_vals, Da0_list, default_colors[:len(k_vals)]):
    w = omega_plus_dim(q_auto, M0=M0_fixed, k_val=k_val)
    axC.plot(q_auto, w.real, color=col, lw=1.8, label=rf"$Da_0 = {Da0}$")
    axC.plot(q_auto, w.imag, "--", color=col, lw=1.0)

axC.axhline(0, ls=":", color="grey", lw=1.0)
axC.set_xlim(0, 0.6)
axC.set_ylim(-0.1, 0.15)
axC.set_xlabel(r"wavenumber $q$")
axC.set_ylabel(r"growth rate $\omega$")
axC.legend(loc="lower left", 
           bbox_to_anchor=(0.03, 0.03), 
           ncol=2, frameon=True, 
           framealpha=0.75, edgecolor="none", columnspacing=0.8, handlelength=1.6, handletextpad=0.5)


# d: Minimal MIPS varying PeC
q_min = np.linspace(1e-3, 1.3, 1000)

PeC_list = [0.2, 0.5, 1.0, 1.5]

for PeC, col in zip(PeC_list, default_colors[:len(PeC_list)]):
    w_plus, _ = omega_pm(q_min, alpha0_base, Da0_base, PeC)
    axD.plot(q_min, w_plus.real, color=col, lw=2.0, label=rf"$Pe_C = {PeC}$")
    axD.plot(q_min, w_plus.imag, "--", color=col, lw=1.0)

axD.axhline(0, ls=":", color="grey", lw=1.0)
axD.set_xlim(0, 1.3)
axD.set_ylim(-0.45, 0.9)
axD.set_xlabel(r"wavenumber $q$")
axD.set_ylabel(r"growth rate $\omega$")
axD.legend(loc="lower center", 
           bbox_to_anchor=(0.5, 0.03), 
           ncol=2, frameon=True, 
           framealpha=0.75, edgecolor="none", columnspacing=0.8, handlelength=1.6, handletextpad=0.5)


# e: Minimal MIPS varying Da0
Da0_list_min = [0.1, 0.3, 0.5, 0.7]

for Da0, col in zip(Da0_list_min, default_colors[:len(Da0_list_min)]):
    w_plus, _ = omega_pm(q_min, alpha0_base, Da0, PeC_base)
    axE.plot(q_min, w_plus.real, color=col, lw=2.0, label=rf"$Da_0 = {Da0}$")
    axE.plot(q_min, w_plus.imag, "--", color=col, lw=1.0)

axE.axhline(0, ls=":", color="grey", lw=1.0)
axE.set_xlim(0, 1.3)
axE.set_ylim(-0.17, 0.56)
axE.set_xlabel(r"wavenumber $q$")
axE.set_ylabel(r"growth rate $\omega$")
axE.legend(loc="lower left", 
           bbox_to_anchor=(0.03, 0.005), 
           ncol=2, frameon=True, 
           framealpha=0.75, edgecolor="none", columnspacing=0.8, handlelength=1.6, handletextpad=0.5)

add_panel_label(axA, "a", x=-0.14, y=1.03)
add_panel_label(axB, "b", x=-0.28, y=1.08)
add_panel_label(axC, "c", x=-0.28, y=1.08)
add_panel_label(axD, "d", x=-0.24, y=1.08)
add_panel_label(axE, "e", x=-0.24, y=1.08)

plt.show()